In [ ]:
pip install findspark

In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1433

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [1]:
import os
import findspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") # Added confirmation message

Findspark initialized successfully.


In [2]:
import os
import findspark
from pyspark.sql import SparkSession
import pyspark

# --- Set the SPARK_HOME environment variable manually ---
# Use forward slashes or double backslashes in the path
os.environ["SPARK_HOME"] = "C:/spark" 

findspark.init()
print("Findspark initialized successfully.") 

# Define the paths to your local JAR files, now with the file:/// scheme
# Use forward slashes exclusively in the final path string
mssql_jar_path = r"file:///C:/spark_drivers/mssql-jdbc-13.2.1.jre11.jar"
mysql_jar_path = r"file:///C:/spark_drivers/mysql-connector-j-9.5.0.jar"

spark = SparkSession.builder \
    .appName("SQLServerToMySQLTransfer") \
    .config("spark.jars", f"{mssql_jar_path},{mysql_jar_path}") \
    .getOrCreate()

print("Spark session created.")

# You can now proceed with your data transfer logic here
# For example:
# df = spark.read.format("jdbc").option(...)

Findspark initialized successfully.
Spark session created.


In [3]:
# SQL Server connection properties
# cannot make partition with char column
import time
sql_server_url = r"jdbc:sqlserver://HYPE\SQL_2022:1433;databaseName=sttwk_2022a;encrypt=true;trustServerCertificate=true"
sqlserver_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchSize": "1000"
}
# Start the timer
start_time = time.time()

# Parameters for parallel read (adjust as needed for your table and data)
# The partitionColumn must be a numeric, date, or timestamp column.
#partition_column = "week" 
#lower_bound = 1
#upper_bound = 13 # Max ID in your table, e.g.
#num_read_partitions = 13 # Read in 10 partitions
    #.option("url", sqlserver_jdbc_url) \
    #.option("partitionColumn", partition_column) \
    #.option("lowerBound", lower_bound) \
    #.option("upperBound", upper_bound) \
    #.option("numPartitions", num_read_partitions) \
sql_server_df = spark.read.format("jdbc") \
    .option("url", sql_server_url) \
    .option("dbtable", "sttwk") \
    .option("user", sqlserver_properties["user"]) \
    .option("password", sqlserver_properties["password"]) \
    .option("driver", sqlserver_properties["driver"]) \
    .load()
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

# Optional: Check the number of partitions created
print(f"DataFrame has {sql_server_df.rdd.getNumPartitions()} partitions after read.")


Total time taken for transfer: 1.40 seconds
DataFrame has 1 partitions after read.


In [4]:
# 10 seconds
from pyspark.sql import functions as sf
# Start the timer
start_time = time.time()

# Option A: Repartition by the character column (hash partitioning)
# The data associated with each unique value in 'char_column' will go to a single partition.
df_to_write = sql_server_df.repartition(13, "week")

# Option B: Repartition generally (even hash distribution)
# Use this if you don't care about co-locating data by the char column, 
# or if the data in 'your_char_column_name' is very skewed.
# df_to_write = sql_server_df.repartition(10) 

print(f"DataFrame has {df_to_write.rdd.getNumPartitions()} partitions before write.")

# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

DataFrame has 13 partitions before write.
Total time taken for transfer: 18.79 seconds


In [5]:
# MySQL connection properties
# Start the timer
start_time = time.time()
# Connection details for MySQL (Destination)
mysql_url = "jdbc:mysql://localhost:3306/AdventureSales"
mysql_user = "root"
mysql_password = "" # Blank password for XAMPP default
mysql_table = "sttwk1"

mysql_jdbc_url = "jdbc:mysql://localhost:3306/AdventureSales?rewriteBatchedStatements=true"
mysql_properties = {
    "user": "<mysql_user>",
    "password": "<mysql_password>",
    "driver": "com.mysql.cj.jdbc.Driver", # Use cj driver for modern versions
    "batchsize": "10000" # Increase batch size
}

# Write the repartitioned DataFrame to the MySQL table
df_to_write.write.format("jdbc") \
    .option("url", mysql_url) \
    .option("dbtable", mysql_table) \
    .option("user", mysql_user) \
    .option("password", mysql_password) \
    .option("driver", mysql_properties["driver"]) \
    .option("batchsize", mysql_properties["batchsize"]) \
    .mode("append") \
    .save()


print("Data successfully inserted into oracle table using 52 parallel processes.")
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")


Data successfully inserted into MySQL table using 13 parallel processes.
Total time taken for transfer: 327.66 seconds


In [7]:
# Assuming 'spark' session is still active from Cell 1
# Start the timer
start_time = time.time()
# Connection details for MySQL
mysql_url = "jdbc:mysql://localhost:3306/AdventureSales"
mysql_user = "root"
mysql_password = "" 

# Define a custom query to read only the top 5 records
# This pushes the LIMIT clause down to the MySQL server itself
mysql_query = "(select year,week,sum(net_qty) from sttwk group by year,week order by year,week) AS data_count"

# Read a limited selection of data into a PySpark DataFrame
df_sample = spark.read.format("jdbc").options(
    url=mysql_url,
    dbtable=mysql_query,
    user=mysql_user,
    password=mysql_password,
    driver="com.mysql.cj.jdbc.Driver"
).load()

print("Displaying count of  records from MySQL table 'sttwk':")
# Use .show() to display the records
df_sample.show(n=1000, truncate=False)

# To get the count of the sample:
print(f"Number of sample rows read: {df_sample.count()}")
# Stop the timer
end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken for transfer: {total_time:.2f} seconds")

Displaying count of  records from MySQL table 'sttwk':
+----+----+------------+
|year|week|sum(net_qty)|
+----+----+------------+
|2022|01  |1176184.4970|
|2022|02  |1450239.3370|
|2022|03  |1337833.0700|
|2022|04  |1625286.4900|
|2022|05  |878449.0460 |
|2022|06  |1382949.0510|
|2022|07  |1459358.2600|
|2022|08  |1767098.0740|
|2022|09  |840785.0730 |
|2022|10  |1462618.2410|
|2022|11  |1452046.7130|
|2022|12  |1754371.8310|
|2022|13  |2204147.0730|
|2022|14  |978069.7020 |
|2022|15  |1163259.9000|
|2022|16  |1272122.0550|
|2022|17  |1565855.1780|
|2022|18  |125007.7630 |
|2022|19  |1255309.4240|
|2022|20  |1341211.9380|
|2022|21  |1611366.0010|
|2022|22  |838319.5850 |
|2022|23  |1436985.5590|
|2022|24  |1263733.2030|
|2022|25  |1288682.5930|
|2022|26  |1185235.8260|
|2022|27  |1111092.9400|
|2022|28  |939130.3910 |
|2022|29  |1123679.3840|
|2022|30  |1343504.3700|
|2022|31  |761249.0800 |
|2022|32  |1175007.4740|
|2022|33  |1137668.0230|
|2022|34  |1820735.9060|
|2022|35  |945395.59